# Continual-learning experiment 


In [1]:
# Cell 1 — imports

from dataclasses import dataclass
from collections.abc import Callable, Iterable
import math
import re
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import hdbscan
import umap
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler
from transformers import AutoModel, AutoTokenizer

from dataset_utils.constants import Cresci17SetTypes
from dataset_utils.Cresci17 import Cresci17
from dataset_utils.Twibot20 import Twibot20
from dataset_utils.InterleavedIterableDataset import InterleavedIterableDataset

from label_remapper import LabelRemapper
from continual_experiment_manager import ContinualExperimentManager
from ClassificationHeads import ProgressiveNeuralNetworkClassifier
from pipeline_utils import train_classifier


c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2 — configuration and frozen DistilBERT

DATASET_ROOT = "./datasets"

MAX_TWEETS_PER_USER = 20
TWEET_BATCH_SIZE = 32
MAX_TOKEN_LENGTH = 128

UMAP_COMPONENTS = 15
UMAP_NEIGHBORS = 20

HDBSCAN_MIN_CLUSTER_SIZE = 10
HDBSCAN_CURRENT_FRACTION = 0.80

EPOCHS = 15
LEARNING_RATE = 1e-2

# Ground-truth intervention is useful for measuring continual classification
# independently from novelty-detection errors.
USE_INTERVENTION_OVERRIDE = True

REPLAY_PER_CLASS = 200
EVAL_BATCH_SIZE = 512
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

bert_model.eval()
for parameter in bert_model.parameters():
    parameter.requires_grad_(False)

scaler = RobustScaler(with_centering=False)
dim_reducer = umap.UMAP(
    n_components=UMAP_COMPONENTS,
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_SEED,
)


Device: cuda


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6513.30it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Cell 3 — task definitions

@dataclass(frozen=True)
class TaskDefinition:
    name: str
    train_factory: Callable[[], Iterable]
    test_factory: Callable[[], Iterable]
    label_transform: Callable[[str], str]


def identity_label(label: str) -> str:
    return str(label)


CRESCI_HUMAN_LABEL = str(Cresci17SetTypes.GENUINE_USER.value)


def twibot_label_transform(label: str) -> str:
    """
    Twibot's human samples are mapped to the already known human class.
    Twibot's generic bot samples become one new class.
    """
    normalized = str(label).strip().lower()

    if normalized == "human":
        return CRESCI_HUMAN_LABEL

    if normalized == "bot":
        return "twibot20_bot"

    raise ValueError(f"Unexpected Twibot20 label: {label!r}")


def make_initial_cresci(mode: str):
    return InterleavedIterableDataset(
        datasets=[
            Cresci17(
                subset_type=Cresci17SetTypes.GENUINE_USER,
                mode=mode,
                root=DATASET_ROOT,
            ),
            Cresci17(
                subset_type=Cresci17SetTypes.FAKE_FOLLOWER,
                mode=mode,
                root=DATASET_ROOT,
            ),
        ],
        mode="RoundRobin",
    )


def make_cresci_task(subset_type: Cresci17SetTypes) -> TaskDefinition:
    return TaskDefinition(
        name=f"Cresci17_{subset_type.name}",
        train_factory=lambda subset=subset_type: Cresci17(
            subset_type=subset,
            mode="train",
            root=DATASET_ROOT,
        ),
        test_factory=lambda subset=subset_type: Cresci17(
            subset_type=subset,
            mode="test",
            root=DATASET_ROOT,
        ),
        label_transform=identity_label,
    )


# Start with these two incremental Cresci classes.
# Add the remaining enum members here after this smaller experiment works.
ACTIVE_CRESCI_SUBSETS = [
    Cresci17SetTypes.SOCIAL_SPAM_1,
    Cresci17SetTypes.TRADITIONAL_SPAM_1,
]

TASK_DEFINITIONS = [
    TaskDefinition(
        name="Cresci17_Initial_Genuine_vs_FakeFollower",
        train_factory=lambda: make_initial_cresci("train"),
        test_factory=lambda: make_initial_cresci("test"),
        label_transform=identity_label,
    ),
    *[make_cresci_task(subset) for subset in ACTIVE_CRESCI_SUBSETS],
    TaskDefinition(
        name="Twibot20_Human_vs_NewBot",
        train_factory=lambda: Twibot20(mode="train", root=DATASET_ROOT),
        test_factory=lambda: Twibot20(mode="test", root=DATASET_ROOT),
        label_transform=twibot_label_transform,
    ),
]

for index, task in enumerate(TASK_DEFINITIONS):
    print(index, task.name)


0 Cresci17_Initial_Genuine_vs_FakeFollower
1 Cresci17_SOCIAL_SPAM_1
2 Cresci17_TRADITIONAL_SPAM_1
3 Twibot20_Human_vs_NewBot


In [4]:
# Cell 4 — embedding helpers for the new Sample dataclass

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)


def safe_float(value, default: float = 0.0) -> float:
    if value is None:
        return default

    try:
        result = float(value)
    except (TypeError, ValueError):
        return default

    if not math.isfinite(result):
        return default

    return result


def safe_log1p(value) -> float:
    return math.log1p(max(safe_float(value), 0.0))


def create_profile_vector(user) -> torch.Tensor:
    values = [
        len(str(user.name or "")),
        len(str(user.screen_name or "")),
        safe_log1p(user.statuses_count),
        safe_log1p(user.followers_count),
        safe_log1p(user.friends_count),
        safe_log1p(user.favourites_count),
        0.0,  # protected is unavailable in the current common dataclass
        float(bool(user.verified)),
    ]

    return torch.tensor(values, dtype=torch.float32)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1.0)
    return summed / counts


def embed_split(
    dataset: Iterable,
    task_name: str,
    label_transform: Callable[[str], str],
    max_samples: int | None = None,
):
    features = []
    labels = []

    print(f"Embedding '{task_name}'...")

    with torch.inference_mode():
        for sample_index, sample in enumerate(dataset):
            if max_samples is not None and sample_index >= max_samples:
                break

            if sample_index % 100 == 0:
                print(f"  {sample_index} users")

            profile_vector = create_profile_vector(sample.user_data)

            texts = []
            for tweet in sample.tweet_data[:MAX_TWEETS_PER_USER]:
                text = str(tweet.text or "").strip()
                if text:
                    texts.append(URL_PATTERN.sub(" url ", text))

            if not texts:
                user_tweet_vector = torch.zeros(
                    bert_model.config.hidden_size,
                    dtype=torch.float32,
                )
            else:
                tweet_batches = []

                for start in range(0, len(texts), TWEET_BATCH_SIZE):
                    batch_texts = texts[start:start + TWEET_BATCH_SIZE]

                    tokens = tokenizer(
                        batch_texts,
                        padding=True,
                        truncation=True,
                        max_length=MAX_TOKEN_LENGTH,
                        return_tensors="pt",
                    )
                    tokens = {
                        name: tensor.to(device)
                        for name, tensor in tokens.items()
                    }

                    outputs = bert_model(**tokens)
                    pooled = mean_pool(
                        outputs.last_hidden_state,
                        tokens["attention_mask"],
                    )
                    tweet_batches.append(pooled.cpu())

                user_tweet_vector = torch.cat(
                    tweet_batches,
                    dim=0,
                ).mean(dim=0)

            combined = torch.cat(
                [profile_vector, user_tweet_vector.float()],
                dim=0,
            )

            features.append(combined.numpy().astype(np.float32))
            labels.append(label_transform(str(sample.label)))

    if not features:
        feature_dim = 8 + bert_model.config.hidden_size
        return (
            np.empty((0, feature_dim), dtype=np.float32),
            [],
        )

    result = np.stack(features).astype(np.float32)
    print(f"Finished '{task_name}': {result.shape}, labels={sorted(set(labels))}")
    return result, labels


In [5]:
# Cell 5 — replay, novelty detection and evaluation helpers

def transform_features(raw_features: np.ndarray) -> np.ndarray:
    scaled = scaler.transform(raw_features)
    reduced = dim_reducer.transform(scaled)
    return np.asarray(reduced, dtype=np.float32)


def replay_to_arrays(replay_by_label):
    all_features = []
    all_labels = []

    for label, class_features in replay_by_label.items():
        if len(class_features) == 0:
            continue

        all_features.append(class_features)
        all_labels.extend([label] * len(class_features))

    if not all_features:
        return (
            np.empty((0, UMAP_COMPONENTS), dtype=np.float32),
            [],
        )

    return np.vstack(all_features), all_labels


def update_replay(
    replay_by_label,
    features: np.ndarray,
    native_labels: list[str],
):
    native_array = np.asarray(native_labels, dtype=object)
    rng = np.random.default_rng(RANDOM_SEED)

    for label in sorted(set(native_labels)):
        new_class_features = features[native_array == label]

        if label in replay_by_label:
            candidate_features = np.vstack(
                [replay_by_label[label], new_class_features]
            )
        else:
            candidate_features = new_class_features

        if len(candidate_features) > REPLAY_PER_CLASS:
            indexes = rng.choice(
                len(candidate_features),
                size=REPLAY_PER_CLASS,
                replace=False,
            )
            candidate_features = candidate_features[indexes]

        replay_by_label[label] = candidate_features.astype(np.float32)


def detect_current_dominated_cluster(
    reference_features: np.ndarray,
    current_features: np.ndarray,
):
    """
    HDBSCAN is run on old reference samples plus the incoming batch.
    A cluster dominated by incoming samples is treated as a novelty signal.
    """
    if len(reference_features) == 0:
        return False, np.full(len(current_features), -1, dtype=np.int64), current_features

    combined = np.vstack([reference_features, current_features])
    is_current = np.concatenate([
        np.zeros(len(reference_features), dtype=bool),
        np.ones(len(current_features), dtype=bool),
    ])

    min_cluster_size = min(
        HDBSCAN_MIN_CLUSTER_SIZE,
        max(2, len(combined) // 4),
    )

    detector = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min(5, min_cluster_size),
        prediction_data=True,
    )
    cluster_labels = detector.fit_predict(combined)

    detected_new = False

    for cluster_id in sorted(set(cluster_labels)):
        if cluster_id == -1:
            continue

        member_mask = cluster_labels == cluster_id
        cluster_size = int(member_mask.sum())
        current_count = int((member_mask & is_current).sum())

        if cluster_size == 0:
            continue

        current_fraction = current_count / cluster_size

        if (
            current_count >= min_cluster_size
            and current_fraction >= HDBSCAN_CURRENT_FRACTION
        ):
            detected_new = True
            break

    return detected_new, cluster_labels, combined


def evaluate_classifier(
    classifier,
    features: np.ndarray,
    labels: list[int],
):
    dataset = TensorDataset(
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)

    classifier.eval()
    predictions = []
    ground_truths = []

    with torch.inference_mode():
        for batch_features, batch_labels in loader:
            logits = classifier(batch_features.to(device))
            batch_predictions = logits.argmax(dim=1).cpu().numpy()

            predictions.extend(batch_predictions.tolist())
            ground_truths.extend(batch_labels.numpy().tolist())

    return ground_truths, predictions


In [6]:
# Cell 6 — experiment objects

label_remapper = LabelRemapper()

experiment_manager = ContinualExperimentManager(
    num_tasks=len(TASK_DEFINITIONS),
    use_intervention_override=USE_INTERVENTION_OVERRIDE,
)

criterion = nn.CrossEntropyLoss()

classifier = None
known_native_labels = set()

# Class-balanced training-memory buffer.
replay_by_label = {}

# Fixed transformed test splits for all tasks already encountered.
test_splits_cache = []


In [7]:
from collections import defaultdict


BALANCED_SAMPLES_PER_CLASS = 200


def create_balanced_training_set(
    current_features,
    current_labels,
    replay_by_label,
    samples_per_class=BALANCED_SAMPLES_PER_CLASS,
):
    rng = np.random.default_rng(RANDOM_SEED)

    feature_pools = defaultdict(list)

    # Add stored replay samples.
    for label, features in replay_by_label.items():
        if len(features) > 0:
            feature_pools[label].append(
                np.asarray(features, dtype=np.float32)
            )

    # Add current-task samples.
    current_labels_array = np.asarray(
        current_labels,
        dtype=object,
    )

    for label in set(current_labels):
        class_features = current_features[
            current_labels_array == label
        ]

        if len(class_features) > 0:
            feature_pools[label].append(class_features)

    balanced_features = []
    balanced_labels = []

    for label in sorted(feature_pools):
        available_features = np.vstack(feature_pools[label])

        # Classes with fewer samples are sampled with replacement.
        replace = len(available_features) < samples_per_class

        selected_indices = rng.choice(
            len(available_features),
            size=samples_per_class,
            replace=replace,
        )

        selected_features = available_features[selected_indices]

        balanced_features.append(selected_features)
        balanced_labels.extend(
            [label] * samples_per_class
        )

    balanced_features = np.vstack(
        balanced_features
    ).astype(np.float32)

    return balanced_features, balanced_labels

In [8]:
# Cell 7 — continual-learning loop

for task_index, task in enumerate(TASK_DEFINITIONS):
    print("\n" + "=" * 72)
    print(f"STEP {task_index}: {task.name}")
    print("=" * 72)

    # ---------------------------------------------------------
    # 1. Embed current training split
    # ---------------------------------------------------------
    raw_train, native_train_labels = embed_split(
        dataset=task.train_factory(),
        task_name=f"{task.name}/train",
        label_transform=task.label_transform,
    )

    if len(native_train_labels) == 0:
        print("No training samples; skipping this task.")
        experiment_manager.advance_to_next_task()
        continue

    native_train_labels = [str(label) for label in native_train_labels]

    # ---------------------------------------------------------
    # 2. Fit representation only on Task 0; freeze afterwards
    # ---------------------------------------------------------
    if task_index == 0:
        scaled_train = scaler.fit_transform(raw_train)
        umap_train = dim_reducer.fit_transform(scaled_train)
        umap_train = np.asarray(umap_train, dtype=np.float32)
    else:
        umap_train = transform_features(raw_train)

    current_labels = set(native_train_labels)
    new_labels = sorted(current_labels - known_native_labels)

    print("Current labels:", sorted(current_labels))
    print("Unseen labels:", new_labels)

    # ---------------------------------------------------------
    # 3. HDBSCAN novelty test against known reference samples
    # ---------------------------------------------------------
    replay_features, replay_native_labels = replay_to_arrays(
        replay_by_label
    )

    if task_index == 0:
        detected_new_cluster = False
        cluster_labels = np.full(
            len(umap_train),
            -1,
            dtype=np.int64,
        )
    else:
        (
            detected_new_cluster,
            cluster_labels,
            _,
        ) = detect_current_dominated_cluster(
            replay_features,
            umap_train,
        )

        combined_ground_truth = (
            replay_native_labels + native_train_labels
        )

        # Keeps your manager's cluster-quality/intervention reporting.
        experiment_manager.check_clusterer_and_expand(
            clusterer_labels=cluster_labels,
            ground_truth_labels=combined_ground_truth,
        )

    print("HDBSCAN novelty signal:", detected_new_cluster)

    # ---------------------------------------------------------
    # 4. Initialize or expand the PNN
    # ---------------------------------------------------------
    if task_index == 0:
        if len(new_labels) < 2:
            raise RuntimeError(
                f"Task 0 must contain at least two labels; got {new_labels}"
            )

        for label in new_labels:
            label_remapper.register(label)

        known_native_labels.update(new_labels)

        classifier = ProgressiveNeuralNetworkClassifier(
            in_features=UMAP_COMPONENTS,
            output_dim=len(new_labels),
            dropout_p=0.1,
        ).to(device)

        trainable_parameters = list(classifier.parameters())

    else:
        should_expand = (
            len(new_labels) > 0
            if USE_INTERVENTION_OVERRIDE
            else detected_new_cluster
        )

        if new_labels and not should_expand:
            raise RuntimeError(
                "The incoming task has unseen labels, but HDBSCAN did not "
                "request expansion. Enable intervention override while "
                "debugging, or add an operator/pseudo-label step."
            )

        if should_expand and new_labels:
            for label in new_labels:
                label_remapper.register(label)

            known_native_labels.update(new_labels)

            trainable_parameters = list(
                classifier.expand_classifier(
                    num_to_add=len(new_labels)
                )
            )

            print(
                f"Expanded classifier by {len(new_labels)} output(s): "
                f"{new_labels}"
            )
        else:
            trainable_parameters = [
                parameter
                for parameter in classifier.parameters()
                if parameter.requires_grad
            ]

    # Do not rely only on classifier.num_classes; test the actual output.
    classifier.eval()
    with torch.inference_mode():
        probe = torch.tensor(
            umap_train[: min(2, len(umap_train))],
            dtype=torch.float32,
            device=device,
        )
        actual_output_count = classifier(probe).shape[1]

    expected_output_count = len(known_native_labels)

    if actual_output_count != expected_output_count:
        raise RuntimeError(
            "PNN expansion mismatch: "
            f"forward() returns {actual_output_count} logits, "
            f"but {expected_output_count} labels are registered. "
            "Check ProgressiveNeuralNetworkClassifier.forward() and "
            "expand_classifier()."
        )

    print("Actual classifier outputs:", actual_output_count)

    # ---------------------------------------------------------
    # 5. Train with old replay examples plus current examples
    # ---------------------------------------------------------
    classifier_train_features, classifier_train_native_labels = (
        create_balanced_training_set(
            current_features=umap_train,
            current_labels=native_train_labels,
            replay_by_label=replay_by_label,
            samples_per_class=BALANCED_SAMPLES_PER_CLASS,
        )
    )

    print("Balanced classifier-training counts:")

    for label in sorted(set(classifier_train_native_labels)):
        count = classifier_train_native_labels.count(label)
        print(f"  {label}: {count}")
        
    classifier_train_labels = label_remapper.convert(
    classifier_train_native_labels
    )

    if trainable_parameters:
        optimizer = torch.optim.Adam(
            trainable_parameters,
            lr=LEARNING_RATE,
        )

        classifier.train()
        train_classifier(
            classifier,
            criterion,
            optimizer,
            classifier_train_features,
            classifier_train_labels,
            epochs=EPOCHS,
            device=device,
        )
    else:
        print("No trainable parameters for this step.")

    # Update replay after training.
    update_replay(
        replay_by_label,
        umap_train,
        native_train_labels,
    )

    # ---------------------------------------------------------
    # 6. Embed and cache this task's fixed test split
    # ---------------------------------------------------------
    raw_test, native_test_labels = embed_split(
        dataset=task.test_factory(),
        task_name=f"{task.name}/test",
        label_transform=task.label_transform,
    )

    native_test_labels = [str(label) for label in native_test_labels]

    unknown_test_labels = (
        set(native_test_labels) - known_native_labels
    )
    if unknown_test_labels:
        raise RuntimeError(
            f"Unregistered test labels: {unknown_test_labels}"
        )

    umap_test = transform_features(raw_test)
    remapped_test_labels = label_remapper.convert(
        native_test_labels
    )

    test_splits_cache.append(
        (umap_test, remapped_test_labels, task.name)
    )

    # ---------------------------------------------------------
    # 7. Evaluate every task encountered so far
    # ---------------------------------------------------------
    print(f"Evaluating {len(test_splits_cache)} test task(s)...")

    for evaluated_task_index, (
        cached_features,
        cached_labels,
        cached_name,
    ) in enumerate(test_splits_cache):

        ground_truths, predictions = evaluate_classifier(
            classifier,
            cached_features,
            cached_labels,
        )

        accuracy = accuracy_score(
            ground_truths,
            predictions,
        )
        macro_f1 = f1_score(
            ground_truths,
            predictions,
            average="macro",
            zero_division=0,
        )

        print(
            f"  Task {evaluated_task_index} — {cached_name}: "
            f"accuracy={accuracy:.4f}, macro-F1={macro_f1:.4f}"
        )

        experiment_manager.record_accuracy(
            task_index=evaluated_task_index,
            accuracy=accuracy,
        )

    experiment_manager.advance_to_next_task()

experiment_manager.summary()



STEP 0: Cresci17_Initial_Genuine_vs_FakeFollower
Embedding 'Cresci17_Initial_Genuine_vs_FakeFollower/train'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
  700 users
  800 users
  900 users
  1000 users
  1100 users
  1200 users
  1300 users
  1400 users
  1500 users
  1600 users
  1700 users
  1800 users
  1900 users
  2000 users
  2100 users
  2200 users
  2300 users
  2400 users
  2500 users
  2600 users
  2700 users
  2800 users
  2900 users
  3000 users
  3100 users
  3200 users
  3300 users
  3400 users
  3500 users
  3600 users
  3700 users
  3800 users
  3900 users
  4000 users
  4100 users
  4200 users
  4300 users
  4400 users
  4500 users
  4600 users
  4700 users
  4800 users
  4900 users
Finished 'Cresci17_Initial_Genuine_vs_FakeFollower/train': (4911, 776), labels=['fake_followers', 'genuine_user']


c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Current labels: ['fake_followers', 'genuine_user']
Unseen labels: ['fake_followers', 'genuine_user']
HDBSCAN novelty signal: False
Actual classifier outputs: 2
Balanced classifier-training counts:
  fake_followers: 200
  genuine_user: 200
Epoch [1/15] - Loss: 2.5688
Epoch [2/15] - Loss: 0.8038
Epoch [3/15] - Loss: 0.5598
Epoch [4/15] - Loss: 0.5063
Epoch [5/15] - Loss: 0.4845
Epoch [6/15] - Loss: 0.3258
Epoch [7/15] - Loss: 0.3583
Epoch [8/15] - Loss: 0.2855
Epoch [9/15] - Loss: 0.2531
Epoch [10/15] - Loss: 0.2398
Epoch [11/15] - Loss: 0.2620
Epoch [12/15] - Loss: 0.2533
Epoch [13/15] - Loss: 0.2279
Epoch [14/15] - Loss: 0.2506
Epoch [15/15] - Loss: 0.2329
Embedding 'Cresci17_Initial_Genuine_vs_FakeFollower/test'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
Finished 'Cresci17_Initial_Genuine_vs_FakeFollower/test': (604, 776), labels=['fake_followers', 'genuine_user']
Evaluating 1 test task(s)...
  Task 0 — Cresci17_Initial_Genuine_vs_FakeFollower

In [11]:
print(
    "Label mapping:",
    {
        label: label_remapper.convert([label])[0]
        for label in sorted(known_native_labels)
    },
)

print("Known labels:", sorted(known_native_labels))

print(
    "Replay classes:",
    {
        label: len(features)
        for label, features in replay_by_label.items()
    },
)

Label mapping: {'fake_followers': 0, 'genuine_user': 1, 'social_spambots_1': 2, 'traditional_spambots_1': 3, 'twibot20_bot': 4}
Known labels: ['fake_followers', 'genuine_user', 'social_spambots_1', 'traditional_spambots_1', 'twibot20_bot']
Replay classes: {'fake_followers': 200, 'genuine_user': 200, 'social_spambots_1': 73, 'traditional_spambots_1': 200, 'twibot20_bot': 200}
